In [11]:
from datasets import load_dataset
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
from transformers import AutoTokenizer, AutoModel, AutoModelForSeq2SeqLM
from sentence_transformers import SentenceTransformer, util
from transformers import pipeline
import torch

1. Load train.csv using the Hugging Face datasets library (do not use pandas). Use the .map() function to create a new column called combined_text that concatenates the prompt and A columns with a space in between. E.g., prompt_text A_text.
What is the exact character length (total number of string characters using Python's len() function, NOT the number of tokens) of the combined_text string for the row at index 51?
Note: We follow zero-indexing here.

In [3]:
# Load CSV (no pandas)
dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]
print(dataset)
# Create combined_text = prompt + " " + A
dataset = dataset.map(
    lambda x: {"combined_text": x["prompt"] + " " + x["A"]}
)

# Character length of row at index 51
length = len(dataset[51]["combined_text"])

print(f"Exact character length of the combined_text string \
for the row at index 51 : {length}")

Generating train split: 0 examples [00:00, ? examples/s]

Dataset({
    features: ['id', 'prompt', 'A', 'B', 'C', 'D', 'E', 'answer'],
    num_rows: 2000
})


Map:   0%|          | 0/2000 [00:00<?, ? examples/s]

Exact character length of the combined_text string for the row at index 51 : 614


2. Initialize the bert-base-uncased tokenizer. Look at the tokenizer's configuration properties: what is the exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer? 

In [4]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

print(f"Exact total vocabulary size (the maximum number of unique subword \
tokens the model knows) hardcoded into this tokenizer : {tokenizer.vocab_size}")

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Exact total vocabulary size (the maximum number of unique subword tokens the model knows) hardcoded into this tokenizer : 30522


3. Transformers rely on special tokens to understand sentence boundaries. Using the bert-base-uncased tokenizer from the previous step, extract the exact integer ID assigned to the [SEP] (Separator) token

In [5]:
print(f"Exact integer ID assigned to the \
[SEP] : {tokenizer.sep_token_id}")

Exact integer ID assigned to the [SEP] : 102


4. Using the bert-base-uncased tokenizer, tokenize the entire prompt column of the train dataset simultaneously. Set padding='max_length', truncation=True, max_length=128, and return_tensors='pt' (PyTorch tensors).

   What is the exact geometric shape (dimensions) of the resulting input_ids tensor?

In [6]:
encoded = tokenizer(
                   list(dataset["prompt"]),
                   padding='max_length',
                   truncation=True, 
                   max_length=128,
                   return_tensors='pt')
print(f"Exact geometric shape (dimensions) of the resulting \
input_ids tensor : {encoded["input_ids"].shape}")

Exact geometric shape (dimensions) of the resulting input_ids tensor : torch.Size([2000, 128])


5. A standard bert-base-uncased model has a hidden embedding size of 768 dimensions and uses exactly 12 attention heads in each layer.

   In Transformer architecture, the hidden size is divided equally among the attention heads. What is the exact dimensionality (size) of each individual attention head?  

In [7]:
hidden_embedding_size = 768
attention_heads = 12
# Attention head is divided equally among the attention heads
individual_attention_head_size = hidden_embedding_size//attention_heads
print(f"Eexact dimensionality (size) of each individual \
attention head : {individual_attention_head_size}")

Eexact dimensionality (size) of each individual attention head : 64


6. Load the bert-base-uncased model using AutoModel.from_pretrained(). Tokenize the prompt from row ID 0 using the tokenizer's default settings (do not apply any manual padding or truncation). Pass this tokenized input through the model. Look at the output object.

   What is the exact shape of the last_hidden_state tensor returned?

   Note: We follow zero-indexing here.

In [8]:
# Load tokenizer and model
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")
model = AutoModel.from_pretrained("bert-base-uncased")

# Row index 0 prompt
text = dataset[0]["prompt"]

# Default tokenization (no padding/truncation)
inputs = tokenizer(
    text,
    return_tensors="pt"
)

# Forward pass
outputs = model(**inputs)

print(f"Exact shape of the last_hidden_state tensor \
returned : {outputs.last_hidden_state.shape}")

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Exact shape of the last_hidden_state tensor returned : torch.Size([1, 31, 768])


7. Using the last_hidden_state tensor from the previous question, extract the embedding vector representing the [CLS] token (which is always the token at index 0). What is the sum of the first 5 float values in this [CLS] vector? (Round your answer to 4 decimal places).

In [9]:
cls_embedding = outputs.last_hidden_state[0, 0, :]

# first 5 values
first_5 = cls_embedding[:5]

print(first_5)

# sum and round
answer = round(first_5.sum().item(), 4)

print(f"Sum of the first 5 float values in \
this [CLS] vector : {answer}")

tensor([-0.4677, -0.0754, -0.2019, -0.0071, -0.4480], grad_fn=<SliceBackward0>)
Sum of the first 5 float values in this [CLS] vector : -1.2001


8. Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

   What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places).

In [12]:
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

model = AutoModel.from_pretrained(
    "bert-base-uncased",
    output_attentions=True
)

text = "Light-ion fusion is a technique."

inputs = tokenizer(
    text,
    return_tensors="pt"
)

# Check tokens
tokens = tokenizer.convert_ids_to_tokens(
    inputs["input_ids"][0]
)

print(tokens)

# Find fusion index
fusion_idx = tokens.index("fusion")

print("fusion index:", fusion_idx)

with torch.no_grad():
    outputs = model(**inputs)

# last layer attention
last_attention = outputs.attentions[-1]

print(last_attention.shape)

# batch 0, head 0, CLS token attention to fusion
attention_value = last_attention[0, 0, 0, fusion_idx]

answer = round(attention_value.item(), 4)

print(f"Exact attention weight (a float value) \
that the [CLS] token (token index 0) pays to the word fusion : {answer}")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


['[CLS]', 'light', '-', 'ion', 'fusion', 'is', 'a', 'technique', '.', '[SEP]']
fusion index: 4
torch.Size([1, 12, 10, 10])
Exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion : 0.1025


9. Load bert-base-uncased with the parameter output_attentions=True. Tokenize the exact string "Light-ion fusion is a technique." (ensuring you set return_tensors='pt') and pass it through the model. Extract the attention matrix for the last layer (index -1) and the first attention head (head index 0).

    What is the exact attention weight (a float value) that the [CLS] token (token index 0) pays to the word fusion (you will need to find the specific token index for fusion in the input_ids)? (Round your answer to 4 decimal places). 

In [13]:
# Load dataset (no pandas)
dataset = load_dataset("csv", data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv")["train"]

# Initialize model
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# Get row ID 0
prompt_text = dataset[0]["prompt"]
option_b_text = dataset[0]["B"]

# Generate embeddings
prompt_embedding = model.encode(prompt_text)
option_b_embedding = model.encode(option_b_text)

# Cosine similarity using sentence_transformers.util.cos_sim()
similarity = util.cos_sim(
    prompt_embedding,
    option_b_embedding
)

answer = round(similarity.item(), 4)

print(f"The resulting similarity score : {answer}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

... (4 outputs left out)

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

The resulting similarity score : 0.7658


10. Build two complete ranking pipelines evaluating every row in train.csv.

    Pipeline 1: Use the TF-IDF cosine similarity approach from Milestone 1.
    
    Pipeline 2: Use the sentence-transformers/all-MiniLM-L6-v2 model to generate embeddings for the prompt and all five options. Rank options using cosine similarity to form Top-3 predictions.
    
    First, what is the final MAP@3 score of the all-MiniLM-L6-v2 pipeline across the entire training set?
    
    Second, count the number of questions for which the correct answer is NOT present in the TF-IDF Top-3 predictions BUT IS present in the MiniLM Top-3 predictions. What is this exact resulting count?

In [14]:
# -----------------------------
# Load data
# -----------------------------

dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]


options = ["A", "B", "C", "D", "E"]


# -----------------------------
# MAP@3 function
# -----------------------------

def apk(actual, predicted, k=3):
    predicted = predicted[:k]

    if actual in predicted:
        return 1 / (predicted.index(actual) + 1)

    return 0


def map3(actuals, predictions):
    scores = []

    for actual, pred in zip(actuals, predictions):
        scores.append(apk(actual, pred))

    return np.mean(scores)



# =================================================
# Pipeline 1: TF-IDF cosine similarity
# =================================================

all_text = []

for row in dataset:
    all_text.append(row["prompt"])
    
    for op in options:
        all_text.append(row[op])


tfidf = TfidfVectorizer()

tfidf.fit(all_text)


tfidf_predictions = []


for row in dataset:

    prompt_vec = tfidf.transform(
        [row["prompt"]]
    )

    sims = {}

    for op in options:

        option_vec = tfidf.transform(
            [row[op]]
        )

        sims[op] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]


    ranked = sorted(
        sims,
        key=sims.get,
        reverse=True
    )

    tfidf_predictions.append(
        ranked[:3]
    )


# =================================================
# Pipeline 2: MiniLM embeddings
# =================================================

model = SentenceTransformer(
    "sentence-transformers/all-MiniLM-L6-v2"
)


minilm_predictions = []


for row in dataset:

    prompt_embedding = model.encode(
        row["prompt"],
        convert_to_tensor=True
    )

    option_texts = [
        row[o] for o in options
    ]

    option_embeddings = model.encode(
        option_texts,
        convert_to_tensor=True
    )


    similarities = util.cos_sim(
        prompt_embedding,
        option_embeddings
    )[0]


    top_indices = similarities.argsort(
        descending=True
    )


    ranked_options = [
        options[i]
        for i in top_indices
    ]


    minilm_predictions.append(
        ranked_options[:3]
    )



# =================================================
# Results
# =================================================

answers = dataset["answer"]


minilm_map3 = map3(
    answers,
    minilm_predictions
)


print(
    "MiniLM MAP@3:",
    round(minilm_map3, 3)
)



# Count questions fixed by MiniLM

fixed_count = 0


for actual, tfidf_pred, mini_pred in zip(
    answers,
    tfidf_predictions,
    minilm_predictions
):

    if (
        actual not in tfidf_pred
        and actual in mini_pred
    ):
        fixed_count += 1


print(
    "TF-IDF missed but MiniLM got:",
    fixed_count
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


MiniLM MAP@3: 0.423
TF-IDF missed but MiniLM got: 531


11. Initialize the Hugging Face pipeline for "zero-shot-classification" (it will default to facebook/bart-large-mnli). For the prompt of the 2nd row (index 1), pass Options A, B, and C as the candidate_labels. What is the probability score given to the top-ranked option? (Round to 4 decimal places).

In [15]:
# Load train.csv
dataset = load_dataset(
    "csv",
    data_files="/kaggle/input/competitions/smart-mcq-solver-challenge/train.csv"
)["train"]


# Initialize zero-shot pipeline
classifier = pipeline(
    "zero-shot-classification"
)


# Row index 1 (2nd row)
prompt_text = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]


# Prediction
result = classifier(
    prompt_text,
    candidate_labels
)


print(result)


# Top ranked probability score
top_score = result["scores"][0]

print(f"The probability score given to the top-ranked\
option : {round(top_score, 4)}")

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

{'sequence': 'What is accelerator-based light-ion fusion?', 'labels': ['Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively easy to implement and can be done in an efficient manner, requiring only a vacuum tube, a pair of electrodes, and a high-voltage transformer. Fusion can be observed with as little as 10 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce light-ion fusion reactions. This method is relatively difficult to implement and requires a complex system of vacuum tubes, electrodes, and transformers. Fusion can be observed with as little as 100 kV between the electrodes.', 'Accelerator-based light-ion fusion is a technique that uses particle accelerators to achieve particle kinetic energies sufficient to induce heavy-ion

12. Run the exact same zero-shot classification as the previous question, but this time pass the argument multi_label=True.

    What is the absolute difference between the sum of the 3 probabilities in the previous question (which uses Softmax) and the sum of the 3 probabilities in this question (which uses independent Sigmoids)?

In [16]:
classifier = pipeline(
    "zero-shot-classification"
)


prompt_text = dataset[1]["prompt"]

candidate_labels = [
    dataset[1]["A"],
    dataset[1]["B"],
    dataset[1]["C"]
]


# -----------------------------
# Softmax (previous question)
# -----------------------------

softmax_result = classifier(
    prompt_text,
    candidate_labels
)

softmax_sum = sum(
    softmax_result["scores"]
)


# -----------------------------
# Independent Sigmoid
# -----------------------------

sigmoid_result = classifier(
    prompt_text,
    candidate_labels,
    multi_label=True
)

sigmoid_sum = sum(
    sigmoid_result["scores"]
)


# Difference

difference = abs(
    softmax_sum - sigmoid_sum
)

print("Softmax scores:", softmax_result["scores"])
print("Softmax sum:", softmax_sum)

print("Sigmoid scores:", sigmoid_result["scores"])
print("Sigmoid sum:", sigmoid_sum)

print("Difference:", round(difference, 4))

No model was supplied, defaulted to facebook/bart-large-mnli and revision d7645e1.
Using a pipeline without specifying a model name and revision in production is not recommended.


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Softmax scores: [0.4574531018733978, 0.2750643193721771, 0.2674825191497803]
Softmax sum: 0.9999999403953552
Sigmoid scores: [0.00046927202492952347, 2.063589454337489e-05, 1.9700619304785505e-05]
Sigmoid sum: 0.0005096085387776839
Difference: 0.9995


13. Let's try Generative AI instead of Classification.

    Load a Small Language Model like google/flan-t5-small using the Hugging Face pipeline("text2text-generation"). Construct the following exact string for row index 0: "Question: [prompt]. Is the correct answer A: [A] or B: [B]? Answer with just the letter A or B." Pass this string to the pipeline, setting max_new_tokens=5. What is the exact string output returned by the model?

In [17]:
# Load FLAN-T5
tokenizer = AutoTokenizer.from_pretrained(
    "google/flan-t5-small"
)

model = AutoModelForSeq2SeqLM.from_pretrained(
    "google/flan-t5-small"
)

# Row index 0
row = dataset[0]

prompt = (
    f"Question: {row['prompt']}. "
    f"Is the correct answer A: {row['A']} or B: {row['B']}? "
    f"Answer with just the letter A or B."
)

# Tokenize
inputs = tokenizer(
    prompt,
    return_tensors="pt"
)

# Generate
with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=5
    )

# Decode
generated_text = tokenizer.decode(
    output_ids[0],
    skip_special_tokens=True
)

print("The exact string output returned by the model:", generated_text)

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/308M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/190 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

The exact string output returned by the model: B
